In [1]:
from groq import Groq
import pandas as pd
import json
import time
import os

In [2]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [3]:
CATEGORIES = {
    "restaurant": "queries where someone wants food, dining, eating, meals",
    "cafe": "queries where someone wants coffee, tea, cafe, beverages",
    "pharmacy": "queries where someone needs medicine, drugs, medical store",
    "hospital": "queries where someone needs doctor, emergency, hospital",
    "atm": "queries where someone needs cash, ATM, money withdrawal",
    "fuel": "queries where someone needs petrol, diesel, fuel pump",
    "parking": "queries where someone needs to park a car or bike",
    "other": "queries unrelated to places like weather, music, time"
}

In [6]:
BATCHES = 4      # 4 batches × 25 = 100 per category × 8 = 800 total
PER_BATCH = 25

all_rows = []

for category, description in CATEGORIES.items():
    print(f"\nGenerating: {category}")
    category_queries = []
    
    for batch_num in range(BATCHES):
        print(f"  Batch {batch_num + 1}/{BATCHES}...")
        
        # Pass already collected queries so LLM avoids repeating them
        existing = category_queries[-20:] if category_queries else []
        avoid_hint = f"\nAvoid repeating these: {existing}" if existing else ""
        
        try:
            completion = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                temperature=0.95,  # slightly higher = more variety
                messages=[{
                    "role": "user",
                    "content": f"""Generate exactly {PER_BATCH} realistic natural language search queries.
Category: {category}
Description: {description}

Rules:
- Mix short and long queries
- Include Indian English phrasing (nearby, near me, open now, chahiye)
- Include occasional Hindi-English mix
- Every query must be unique and different from others
- No numbering, no explanations
- Return ONLY a valid JSON array of {PER_BATCH} strings
{avoid_hint}

Example format: ["query one", "query two", ...]"""
                }]
            )
            
            text = completion.choices[0].message.content.strip()
            # Clean markdown if present
            text = text.replace("```json", "").replace("```", "").strip()
            
            queries = json.loads(text)
            
            # Take only unique ones
            for q in queries:
                q = q.strip()
                if q and q not in category_queries:
                    category_queries.append(q)
            
            print(f"    Got {len(queries)}, total so far: {len(category_queries)}")
            time.sleep(1.5)  # avoid rate limit
            
        except Exception as e:
            print(f"    Batch {batch_num + 1} failed: {e}")
            time.sleep(3)
            continue
    
    for q in category_queries:
        all_rows.append({"query": q, "category": category})
    print(f"  Final count for {category}: {len(category_queries)}")


Generating: restaurant
  Batch 1/4...
    Got 25, total so far: 25
  Batch 2/4...
    Got 24, total so far: 49
  Batch 3/4...
    Got 24, total so far: 72
  Batch 4/4...
    Got 24, total so far: 96
  Final count for restaurant: 96

Generating: cafe
  Batch 1/4...
    Got 25, total so far: 25
  Batch 2/4...
    Got 24, total so far: 49
  Batch 3/4...
    Got 24, total so far: 73
  Batch 4/4...
    Got 25, total so far: 98
  Final count for cafe: 98

Generating: pharmacy
  Batch 1/4...
    Got 25, total so far: 25
  Batch 2/4...
    Got 24, total so far: 49
  Batch 3/4...
    Got 24, total so far: 73
  Batch 4/4...
    Got 24, total so far: 96
  Final count for pharmacy: 96

Generating: hospital
  Batch 1/4...
    Got 26, total so far: 26
  Batch 2/4...
    Got 24, total so far: 50
  Batch 3/4...
    Got 23, total so far: 72
  Batch 4/4...
    Got 23, total so far: 94
  Final count for hospital: 94

Generating: atm
  Batch 1/4...
    Got 25, total so far: 25
  Batch 2/4...
    Got 24, 

In [7]:
# Save
df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset=["query"])
df = df.sample(frac=1).reset_index(drop=True)
df.to_csv("place_intent_data.csv", index=False)
print(f"\nTotal saved: {len(df)}")
print(df["category"].value_counts())


Total saved: 766
category
cafe          98
fuel          97
restaurant    96
pharmacy      96
parking       96
atm           95
other         94
hospital      94
Name: count, dtype: int64


In [8]:
import pandas as pd

df = pd.read_csv("place_intent_data.csv")
print(df.shape)
print(df["category"].value_counts())
print(df.sample(10))

(766, 2)
category
cafe          98
fuel          97
restaurant    96
pharmacy      96
parking       96
atm           95
other         94
hospital      94
Name: count, dtype: int64
                                                 query    category
120           is yahan par parking space available hai     parking
27   koi cafe jo tea ke saath dessert bhi serve kar...        cafe
240       part time work from home jobs for housewives       other
434                                        nearby café  restaurant
62                 Paisa chahiye to ATM kaise use kare         atm
221        kaise hum apne goals achieve kar sakte hain       other
59                           dhaba style food open now  restaurant
249                      coffee and light refreshments        cafe
82       emergency hospital in Delhi with good doctors    hospital
702  meri dadi ko hospital mein admit karane ke liy...    hospital


In [9]:
for cat in df["category"].unique():
    print("\n", cat.upper())
    print(df[df["category"] == cat]["query"].sample(10).tolist())


 RESTAURANT
['nearby cafes with snacks', 'koi dhaba jahan main non veg khana kha sakun', 'koi restaurant jo home delivery karti ho', 'main chinese food khaana chahta hoon', 'koi jagah jahan main garlic bread kha sakun', 'food places open now in my area', 'local food joints', 'food places open now in my location', 'best restaurants in my city', 'khaane ke liye koi aur jagah bataiye']

 OTHER
['best online platforms for buying second hand books', 'motivational quotes in hindi', 'kuch behtareen ways to improve concentration', 'types of meditation techniques', 'how to create a successful ecommerce website', 'how to start a small business with low investment', 'best ways to learn a new language', 'salon near me open now', 'kaise hum apne relationships ko strong bana sakte hain', 'how to learn photography skills at home']

 FUEL
['petrol bunk nearby', 'diesel petrol pump nearby', 'fuel station near my current location', 'diesel fuel stations near my office', 'nearby petrol pumps with diesel

In [10]:
import pandas as pd

df = pd.read_csv("place_intent_data.csv")
print("Before:", len(df))

# Fix obvious mislabels — cafe queries sitting in restaurant
cafe_keywords = ["coffee", "cafe", "chai", "tea", "cappuccino", "latte", "espresso"]
restaurant_keywords = ["food", "eat", "meal", "dinner", "lunch", "breakfast", "dosa", "biryani", "restaurant"]

def fix_label(row):
    q = row["query"].lower()
    # cafe keywords in restaurant label → move to cafe
    if row["category"] == "restaurant" and any(k in q for k in cafe_keywords):
        return "cafe"
    # restaurant keywords in cafe label → move to restaurant
    if row["category"] == "cafe" and any(k in q for k in restaurant_keywords):
        return "restaurant"
    return row["category"]

df["category"] = df.apply(fix_label, axis=1)

# Drop rows where 'other' sounds like a place query
place_words = ["near me", "nearby", "nearest", "open now", "find", "where", "location"]
def is_bad_other(row):
    if row["category"] == "other":
        q = row["query"].lower()
        return any(w in q for w in place_words)
    return False

df = df[~df.apply(is_bad_other, axis=1)]

print("After cleanup:", len(df))
print(df["category"].value_counts())
df.to_csv("place_intent_data_clean.csv", index=False)
print("Saved to place_intent_data_clean.csv")

Before: 766
After cleanup: 753
category
restaurant    103
fuel           97
pharmacy       96
parking        96
atm            95
hospital       94
cafe           91
other          81
Name: count, dtype: int64
Saved to place_intent_data_clean.csv


In [12]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])

# Save label mapping — you NEED this later in app.py
import json
label_map = {i: c for i, c in enumerate(le.classes_)}
with open("label_map.json", "w") as f:
    json.dump(label_map, f)

print("Label map:", label_map)
# e.g. {0: 'atm', 1: 'cafe', 2: 'fuel', ...}

Label map: {0: 'atm', 1: 'cafe', 2: 'fuel', 3: 'hospital', 4: 'other', 5: 'parking', 6: 'pharmacy', 7: 'restaurant'}


In [14]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 612, Test: 154


In [15]:
import tensorflow as tf
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts, labels):
    enc = tok(
        list(texts),
        truncation=True,
        padding=True,
        max_length=32,
        return_tensors="tf"
    )
    return tf.data.Dataset.from_tensor_slices(
        (dict(enc), list(labels))
    ).batch(16)

train_ds = tokenize(train_df["query"], train_df["label"])
test_ds  = tokenize(test_df["query"],  test_df["label"])

/Users/keertinayak30/.conda/envs/here-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-04 15:11:47.939554: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-04 15:11:47.939596: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-06-04 15:11:47.939603: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-06-04 15:11:47.939666: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-04 15:11:47.939801: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0

In [16]:
from transformers import TFAutoModelForSequenceClassification

num_labels = len(label_map)  # 8

model = TFAutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.fit(train_ds, validation_data=test_ds, epochs=4)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/4


2026-06-04 15:12:18.905536: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


39/39 [==============================] - ETA: 0s - loss: 2.0473 - accuracy: 0.1912

2026-06-04 15:12:41.729124: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


39/39 [==============================] - 31s 497ms/step - loss: 2.0473 - accuracy: 0.1912 - val_loss: 1.9124 - val_accuracy: 0.3052
Epoch 2/4
39/39 [==============================] - 12s 289ms/step - loss: 1.6252 - accuracy: 0.6585 - val_loss: 1.1866 - val_accuracy: 0.8571
Epoch 3/4
39/39 [==============================] - 12s 291ms/step - loss: 0.8430 - accuracy: 0.9461 - val_loss: 0.5147 - val_accuracy: 0.9481
Epoch 4/4
39/39 [==============================] - 11s 284ms/step - loss: 0.3384 - accuracy: 0.9886 - val_loss: 0.2588 - val_accuracy: 0.9740


In [17]:
import numpy as np
from sklearn.metrics import classification_report

logits = model.predict(test_ds).logits
preds  = np.argmax(logits, axis=1)
labels = test_df["label"].values

print(classification_report(
    labels, preds,
    target_names=list(label_map.values()),
    digits=3
))

2026-06-04 15:13:30.952581: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


10/10 [==============================] - 6s 318ms/step
              precision    recall  f1-score   support

         atm      1.000     1.000     1.000        19
        cafe      0.909     1.000     0.952        20
        fuel      1.000     1.000     1.000        20
    hospital      1.000     1.000     1.000        19
       other      0.947     0.947     0.947        19
     parking      1.000     1.000     1.000        19
    pharmacy      1.000     1.000     1.000        19
  restaurant      0.941     0.842     0.889        19

    accuracy                          0.974       154
   macro avg      0.975     0.974     0.974       154
weighted avg      0.974     0.974     0.974       154



In [18]:
model.save_pretrained("intent_model_v2")
tok.save_pretrained("intent_model_v2")
print("Saved to intent_model_v2/")

Saved to intent_model_v2/


In [19]:
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
import tensorflow as tf
import json
import numpy as np

model = TFAutoModelForSequenceClassification.from_pretrained("intent_model_v2")
tokenizer = AutoTokenizer.from_pretrained("intent_model_v2")

with open("label_map.json") as f:
    label_map = json.load(f)

label_map = {int(k): v for k, v in label_map.items()}

queries = [
    "nearest ATM",
    "petrol pump nearby",
    "need medicines urgently",
    "doctor open now",
    "parking near bandra"
]

for q in queries:
    enc = tokenizer(
        q,
        return_tensors="tf",
        truncation=True,
        padding=True,
        max_length=32
    )

    logits = model(enc).logits
    pred = int(tf.argmax(logits, axis=1)[0])

    print(q)
    print("ID:", pred)
    print("Category:", label_map[pred])
    print()

Some layers from the model checkpoint at intent_model_v2 were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at intent_model_v2.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


nearest ATM
ID: 0
Category: atm

petrol pump nearby
ID: 2
Category: fuel

need medicines urgently
ID: 3
Category: hospital

doctor open now
ID: 3
Category: hospital

parking near bandra
ID: 5
Category: parking

